In [ ]:
# @title Installing
# Cell 1: Install dependencies
!pip install -q omnivoice gradio
!pip install -q torchaudio --extra-index-url https://download.pytorch.org/whl/cu128

In [ ]:
# @title Loading Models
# Cell 2: Load model
import torch
import torchaudio
import gradio as gr
import tempfile
from omnivoice import OmniVoice

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
dtype  = torch.float16 if torch.cuda.is_available() else torch.float32
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)')

print('Loading OmniVoice model (first time downloads ~3-5 GB)...')
model = OmniVoice.from_pretrained('k2-fsa/OmniVoice', device_map=device, dtype=dtype)
print('Model loaded!')

In [ ]:
# @title Final Setup
# Cell 3: Premium Customized OmniVoice UI by Hunaar Ansari

import gradio as gr
import tempfile
import torchaudio
import torch

NONVERBAL_TAGS = ['[laughter]', '[sigh]', '[sniff]', '[question-en]', '[surprise-ah]', '[dissatisfaction-hnn]', '[confirmation-en]']

DESIGN_PRESETS = {
    'Custom (type your own)': '',
    'Female, British accent': 'female, british accent',
    'Male, American accent': 'male, american accent',
    'Female, high pitch, young': 'female, high pitch, young adult',
    'Male, deep, elderly': 'male, very low pitch, elderly',
    'Female, whisper': 'female, whisper',
    'Male, Australian accent': 'male, australian accent',
    'Female, Indian accent': 'female, indian accent',
    'Male, fast, energetic': 'male, high pitch, young adult',
}

EXAMPLE_TEXTS = [
    'Hello! OmniVoice supports over 600 languages with state-of-the-art voice cloning.',
    '[laughter] You really got me. I did not see that coming at all.',
    'Breaking news: scientists have made a remarkable discovery.',
    'Welcome to our podcast. Today we are diving deep into open source AI.',
]

def generate(text, mode, ref_audio, ref_text, design_preset, custom_instruct, num_steps, speed):
    if not text.strip():
        return None, 'Please enter some text.'
    try:
        kwargs = dict(text=text, num_step=int(num_steps), speed=float(speed))
        if mode == 'Voice Cloning':
            if ref_audio is None:
                return None, 'Please upload a reference audio file.'
            kwargs['ref_audio'] = ref_audio
            if ref_text.strip():
                kwargs['ref_text'] = ref_text
        elif mode == 'Voice Design':
            instruct = custom_instruct.strip() if design_preset == 'Custom (type your own)' else DESIGN_PRESETS[design_preset]
            if not instruct:
                return None, 'Please enter a voice design instruction.'
            kwargs['instruct'] = instruct

        audio = model.generate(**kwargs)
        tmp = tempfile.NamedTemporaryFile(suffix='.wav', delete=False)
        audio_tensor = audio[0] if isinstance(audio[0], torch.Tensor) else torch.tensor(audio[0])
        if audio_tensor.dim() == 1:
            audio_tensor = audio_tensor.unsqueeze(0)
        torchaudio.save(tmp.name, audio_tensor.cpu(), 24000)
        duration = audio[0].shape[-1] / 24000
        return tmp.name, f'Generated {duration:.1f}s | Mode: {mode}'
    except Exception as e:
        return None, f'Error: {e}'

def update_mode(mode):
    return (gr.update(visible=mode == 'Voice Cloning'), gr.update(visible=mode == 'Voice Cloning'),
            gr.update(visible=mode == 'Voice Design'), gr.update(visible=mode == 'Voice Design'))

def update_preset(preset):
    return gr.update(visible=preset == 'Custom (type your own)')

def insert_tag(text, tag):
    return text + ' ' + tag

# Modern CSS - Deep Purple + Cyan Theme
css = '''
:root {
    --primary: #7c3aed;
    --accent: #22d3ee;
}

#header {
    text-align: center;
    padding: 20px 0;
    background: linear-gradient(135deg, #1e1b4b, #312e81);
    border-radius: 20px;
    margin-bottom: 20px;
    box-shadow: 0 15px 40px rgba(124, 58, 237, 0.4);
    position: relative;
    overflow: hidden;
}

#header::before {
    content: '';
    position: absolute;
    top: -50%;
    left: -50%;
    width: 200%;
    height: 200%;
    background: radial-gradient(circle, rgba(34,211,238,0.15) 0%, transparent 70%);
    animation: glow 8s infinite alternate;
}

.social-panel {
    display: flex;
    justify-content: center;
    gap: 12px;
    flex-wrap: wrap;
    margin-top: 12px;
    padding: 12px 20px;
    background: rgba(255,255,255,0.08);
    border-radius: 9999px;
    backdrop-filter: blur(12px);
    border: 1px solid rgba(124, 58, 237, 0.3);
}

.social-link {
    padding: 8px 16px;
    background: rgba(255,255,255,0.1);
    border-radius: 9999px;
    color: white;
    text-decoration: none;
    font-size: 0.95em;
    transition: all 0.3s ease;
    display: flex;
    align-items: center;
    gap: 6px;
}

.social-link:hover {
    background: linear-gradient(90deg, var(--primary), var(--accent));
    transform: translateY(-3px) scale(1.08);
    color: white;
}

.gen-btn {
    background: linear-gradient(90deg, var(--primary), var(--accent)) !important;
    color: white !important;
    border-radius: 50px !important;
    padding: 16px 40px !important;
    font-weight: 700;
    box-shadow: 0 10px 30px rgba(124, 58, 237, 0.5);
}

.gen-btn:hover {
    transform: translateY(-4px);
    box-shadow: 0 20px 40px rgba(34, 211, 238, 0.6);
}

@keyframes glow {
    from { opacity: 0.6; }
    to { opacity: 1; }
}
'''

with gr.Blocks(title='OmniVoice - Hunaar Ansari') as demo:

    gr.HTML("""
    <div id="header">
        <h1 style="font-size: 3em; margin: 0 0 8px 0; background: linear-gradient(90deg, #c4b5fd, #67e8f9); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">
            OmniVoice
        </h1>
        <p style="margin: 0; opacity: 0.9;">Advanced Voice Cloning & Design • 600+ Languages</p>
        <p style="margin: 8px 0 0 0; font-size: 1.1em;">by <strong>Hunaar Ansari</strong></p>

        <div class="social-panel">
            <a href="https://www.youtube.com/@Hunaarg1" target="_blank" class="social-link">▶️ YouTube</a>
            <a href="https://whatsapp.com/channel/0029VabTEup4dTnKxHsYP01W" target="_blank" class="social-link">📲 WhatsApp</a>
            <a href="https://www.instagram.com/hunaar_ansari/" target="_blank" class="social-link">📸 Instagram</a>
            <a href="https://t.me/hunaarg" target="_blank" class="social-link">✈️ Telegram</a>
            <a href="https://www.facebook.com/adeel429burewala" target="_blank" class="social-link">📘 Facebook</a>
        </div>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=3):
            text_input = gr.Textbox(label="Text to speak", placeholder="Enter your text here...", lines=5)

            gr.Markdown("**Non-verbal tags:**")
            with gr.Row():
                for tag in NONVERBAL_TAGS:
                    btn = gr.Button(tag, size="sm")
                    btn.click(insert_tag, inputs=[text_input, gr.State(tag)], outputs=text_input)

            mode_radio = gr.Radio(['Auto Voice', 'Voice Cloning', 'Voice Design'], value='Auto Voice', label="Generation Mode")

            ref_audio = gr.Audio(label="Reference Audio", type="filepath", visible=False)
            ref_text = gr.Textbox(label="Reference Transcript (optional)", visible=False)
            design_preset = gr.Dropdown(choices=list(DESIGN_PRESETS.keys()), value="Female, British accent", label="Voice Style", visible=False)
            custom_instruct = gr.Textbox(label="Custom Voice Instruction", visible=False)

            mode_radio.change(update_mode, mode_radio, [ref_audio, ref_text, design_preset, custom_instruct])
            design_preset.change(update_preset, design_preset, custom_instruct)

            with gr.Row():
                steps = gr.Slider(8, 64, 32, step=8, label="Diffusion Steps")
                speed = gr.Slider(0.5, 2.0, 1.0, step=0.1, label="Speed")

            gen_btn = gr.Button("🚀 Generate Speech", elem_classes="gen-btn", variant="primary")
            status = gr.Markdown()

        with gr.Column(scale=2):
            audio_out = gr.Audio(label="Output Audio", type="filepath")
            gr.Markdown("### How to use")

    gr.Markdown("### Examples")
    gr.Examples(examples=[[t, 'Auto Voice', None, '', 'Female, British accent', '', 32, 1.0] for t in EXAMPLE_TEXTS],
                inputs=[text_input, mode_radio, ref_audio, ref_text, design_preset, custom_instruct, steps, speed])

    gen_btn.click(generate, [text_input, mode_radio, ref_audio, ref_text, design_preset, custom_instruct, steps, speed], [audio_out, status])

# Launch
# Launch with prevent_thread_lock to enable automatic discovery & dynamic registration
import os
import requests
app, local_url, share_url = demo.launch(
    share=True,
    debug=True,
    prevent_thread_lock=True,
    theme=gr.themes.Soft(primary_hue="violet", secondary_hue="cyan"),
    css=css
)

active_url = share_url or getattr(demo, "share_url", None) or local_url

print("\n" + "=" * 70)
print("🚀 JARVIS OS OMNIVOICE SERVER INITIALIZED")
print(f"🔗 Public Gradio URL: {active_url}")
print("🔑 Default Token:    jarvis_omnivoice_secret_2026")
print("=" * 70)

# Automatic registration callback to JARVIS OS if callback URL is configured
JARVIS_CALLBACK_URL = os.environ.get("JARVIS_CALLBACK_URL", "").strip()
REGISTRATION_TOKEN = os.environ.get("JARVIS_REGISTRATION_TOKEN", "jarvis_omnivoice_secret_2026").strip()

if JARVIS_CALLBACK_URL and active_url:
    try:
        reg_endpoint = f"{JARVIS_CALLBACK_URL.rstrip('/')}/api/integrations/omnivoice/register"
        print(f"📡 Attempting automatic registration with JARVIS at: {reg_endpoint} ...")
        resp = requests.post(
            reg_endpoint,
            json={
                "gradio_url": active_url,
                "token": REGISTRATION_TOKEN,
                "session_id": "google_colab_t4",
                "verify": True
            },
            timeout=10
        )
        if resp.status_code == 200:
            print("✅ Successfully registered with JARVIS OS!")
        else:
            print(f"⚠️ JARVIS registration returned {resp.status_code}: {resp.text}")
    except Exception as exc:
        print(f"ℹ️ Note: Auto-callback to JARVIS unreachable ({exc}). You can register via Web Dashboard.")
else:
    print("💡 To auto-register, set JARVIS_CALLBACK_URL in your environment or enter the URL in the JARVIS Web Dashboard.")

print("=" * 70 + "\n")

# Keep Gradio running
demo.block_till_done()
